# 02 — Datenqualitätsanalyse

## Kfz Insurance Risk & Pure Premium Intelligence Platform

### Ziel

In diesem Notebook wird die Datenqualität der beiden Rohdatensätze
`freMTPL2freq` und `freMTPL2sev` systematisch untersucht.

Die wichtigsten Prüfungen umfassen:

- fehlende Werte
- Duplikate
- ungültige Exposure-Werte
- auffällige Schadenanzahlen
- auffällige Schadenhöhen
- Konsistenz zwischen Frequency- und Severity-Daten
- verwaiste Severity-Datensätze
- Join-Kardinalität zwischen den beiden Datensätzen

Die Prüfungen werden zunächst manuell und nachvollziehbar durchgeführt.
Anschließend werden die Ergebnisse mit den automatisierten Prüfungen aus
`src/data_quality/checks.py` verglichen.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

FREQUENCY_FILE = RAW_DATA_DIR / "freMTPL2freq.pq"
SEVERITY_FILE = RAW_DATA_DIR / "freMTPL2sev.pq"

In [3]:
print("Project root:", PROJECT_ROOT)
print("Frequency file exists:", FREQUENCY_FILE.exists())
print("Severity file exists:", SEVERITY_FILE.exists())

Project root: c:\Users\Mesut\Desktop\kfz-insurance-risk-platform
Frequency file exists: True
Severity file exists: True


In [4]:
frequency_df = pd.read_parquet(FREQUENCY_FILE)
severity_df = pd.read_parquet(SEVERITY_FILE)

In [5]:
print("Frequency shape:", frequency_df.shape)
print("Severity shape:", severity_df.shape)

Frequency shape: (678013, 12)
Severity shape: (26639, 2)


## 1. Überblick über die Datensätze

Zunächst werden beide Rohdatensätze hinsichtlich Struktur, Spalten und ersten Beobachtungen untersucht.

In [6]:
frequency_df.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1.0,1,0.10,D,5,0,55,50,B12,Regular,1217.0,R82
1,3.0,1,0.77,D,5,0,55,50,B12,Regular,1217.0,R82
2,5.0,1,0.75,B,6,2,52,50,B12,Diesel,54.0,R22
3,10.0,1,0.09,B,7,0,46,50,B12,Diesel,76.0,R72
4,11.0,1,0.84,B,7,0,46,50,B12,Diesel,76.0,R72


In [7]:
severity_df.head()

,IDpol,ClaimAmount
0,1552.0,995.20
1,1010996.0,1128.12
2,4024277.0,1851.11
3,4007252.0,1204.00
4,4046424.0,1204.00


## 2. Struktur der Datensätze

Als Nächstes werden Datentypen, Spalten und Speicherinformationen geprüft.

In [8]:
frequency_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 678013 entries, 0 to 678012
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   IDpol       678013 non-null  float64 
 1   ClaimNb     678013 non-null  uint8   
 2   Exposure    678013 non-null  float64 
 3   Area        678013 non-null  category
 4   VehPower    678013 non-null  uint8   
 5   VehAge      678013 non-null  uint8   
 6   DrivAge     678013 non-null  uint8   
 7   BonusMalus  678013 non-null  uint8   
 8   VehBrand    678013 non-null  category
 9   VehGas      678013 non-null  object  
 10  Density     678013 non-null  float64 
 11  Region      678013 non-null  category
dtypes: category(3), float64(3), object(1), uint8(5)
memory usage: 25.9+ MB


In [9]:
severity_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26639 entries, 0 to 26638
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   IDpol        26639 non-null  float64
 1   ClaimAmount  26639 non-null  float64
dtypes: float64(2)
memory usage: 416.4 KB


## 3. Fehlende Werte

Es wird geprüft, ob in den beiden Datensätzen fehlende Werte vorhanden sind.

In [10]:
print("Frequency:")
print(frequency_df.isna().sum())

print("\nSeverity:")
print(severity_df.isna().sum())

Frequency:
IDpol         0
ClaimNb       0
Exposure      0
Area          0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Density       0
Region        0
dtype: int64

Severity:
IDpol          0
ClaimAmount    0
dtype: int64


**Ergebnis:** In beiden Datensätzen wurden keine fehlenden Werte gefunden.

## 4. Duplikate

Es wird geprüft, ob identische oder mehrfach vorkommende Datensätze vorhanden sind.

In [11]:
print("Frequency exact duplicates:")
print(frequency_df.duplicated().sum())

Frequency exact duplicates:
0


In [12]:
print("Severity exact duplicates:")
print(severity_df.duplicated().sum())

Severity exact duplicates:
255


**Ergebnis:** Im Frequency-Datensatz wurden keine exakten Duplikate gefunden. Im Severity-Datensatz wurden 255 potenzielle Duplikate identifiziert.

In [13]:
severity_df[
    severity_df.duplicated(keep=False)
].sort_values(["IDpol", "ClaimAmount"]).head(20)

,IDpol,ClaimAmount
5022,18698.0,1128.12
6355,18698.0,1128.12
4514,19887.0,1128.12
4521,19887.0,1128.12
6318,24992.0,1128.12
6322,24992.0,1128.12
5862,25763.0,1128.12
5866,25763.0,1128.12
24094,29567.0,1128.12
24096,29567.0,1128.12


**Bewertung:** Die 255 exakten Duplikate im Severity-Datensatz werden als potenzielle Duplikate betrachtet. Da keine eindeutige Schaden-ID vorhanden ist, werden sie nicht entfernt.

### Eindeutigkeit der Policy-IDs

Es wird geprüft, ob jede Policy-ID im Frequency-Datensatz nur einmal vorkommt.

In [14]:
duplicate_policy_ids = frequency_df["IDpol"].duplicated().sum()

print("Duplicate Policy IDs:", duplicate_policy_ids)

Duplicate Policy IDs: 0


**Ergebnis:** Alle Policy-IDs im Frequency-Datensatz sind eindeutig.

## 5. Prüfung der Exposure-Werte

Es wird geprüft, ob die Exposure-Werte im gültigen Bereich liegen.

In [15]:
frequency_df["Exposure"].describe()

count    678013.000000
mean          0.528750
std           0.364442
min           0.002732
25%           0.180000
50%           0.490000
75%           0.990000
max           2.010000
Name: Exposure, dtype: float64

### Ungültige Exposure-Werte
Nun wird geprüft, wie viele Exposure-Werte außerhalb des gültigen Bereichs liegen.

In [16]:
invalid_exposure = frequency_df[
    (frequency_df["Exposure"] <= 0)
    | (frequency_df["Exposure"] > 1)
]

print("Invalid Exposure rows:", len(invalid_exposure))
print("Exposure <= 0:", (frequency_df["Exposure"] <= 0).sum())
print("Exposure > 1:", (frequency_df["Exposure"] > 1).sum())

Invalid Exposure rows: 1224
Exposure <= 0: 0
Exposure > 1: 1224


In [17]:
invalid_exposure.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
179652,1179653.0,0,1.99,A,11,10,50,50,B12,Regular,29.0,R25
179653,1179654.0,1,1.17,D,4,2,21,112,B6,Regular,721.0,R93
179655,1179656.0,1,1.12,E,6,0,37,72,B4,Diesel,9307.0,R82
179657,1179658.0,0,1.48,D,7,4,46,50,B6,Diesel,776.0,R93
179659,1179660.0,1,1.50,C,8,14,26,76,B1,Regular,372.0,R25


**Ergebnis:** 1.224 Datensätze haben eine Exposure über 1. Nicht-positive Exposure-Werte wurden nicht gefunden.

## 6. Prüfung der Schadenanzahl

Die Verteilung der Schadenanzahl pro Police wird untersucht.

In [18]:
frequency_df["ClaimNb"].value_counts().sort_index()

ClaimNb
0     643953
1      32178
2       1784
3         82
4          7
5          2
6          1
8          1
9          1
11         3
16         1
Name: count, dtype: int64

### Auffällige Schadenanzahlen

Policen mit mehr als vier Schäden werden gesondert betrachtet.

In [19]:
high_claim_policies = frequency_df[
    frequency_df["ClaimNb"] > 4
]

print("Policies with ClaimNb > 4:", len(high_claim_policies))

high_claim_policies

Policies with ClaimNb > 4: 9


,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
6539,19471.0,11,1.00,A,5,6,65,50,B2,Diesel,13.0,R24
46470,93954.0,5,1.00,E,7,9,67,50,B2,Diesel,4762.0,R93
304484,2216294.0,6,0.33,D,4,12,52,50,B1,Regular,824.0,R91
320764,2239279.0,8,0.41,D,4,12,52,50,B1,Regular,824.0,R91
321248,2241683.0,16,0.33,D,4,12,52,50,B1,Regular,824.0,R91
321512,2248174.0,9,0.08,D,4,12,52,50,B1,Regular,824.0,R91
344616,2277762.0,5,0.08,D,4,12,52,50,B1,Regular,824.0,R91
487268,3253234.0,11,0.08,D,4,13,53,50,B1,Regular,824.0,R91
488270,3254353.0,11,0.07,D,4,13,53,50,B1,Regular,824.0,R91


**Ergebnis:** 9 Policen weisen mehr als vier Schäden auf. Diese Werte werden als auffällig markiert, aber nicht entfernt.

## 7. Prüfung der Schadenhöhen

Die Verteilung der Schadenbeträge wird auf auffällige Extremwerte untersucht.

In [21]:
severity_df["ClaimAmount"].describe().T

count    2.663900e+04
mean     2.278536e+03
std      2.929748e+04
min      1.000000e+00
25%      6.868100e+02
50%      1.172000e+03
75%      1.228080e+03
max      4.075401e+06
Name: ClaimAmount, dtype: float64

### Extremwerte der Schadenhöhe

Die hohen Maximalwerte deuten auf eine stark rechtsschiefe Verteilung hin.

In [22]:
print("99% Quantile:", severity_df["ClaimAmount"].quantile(0.99))
print("99.9% Quantile:", severity_df["ClaimAmount"].quantile(0.999))
print("Maximum:", severity_df["ClaimAmount"].max())

99% Quantile: 16793.704399999988
99.9% Quantile: 162784.4423601336
Maximum: 4075400.56


**Ergebnis:** Die Schadenhöhen weisen eine stark rechtsschiefe Verteilung mit wenigen sehr hohen Extremwerten auf.

## 8. Konsistenz zwischen Frequency und Severity

Es wird geprüft, ob die gemeldete Schadenanzahl mit den vorhandenen Severity-Einträgen übereinstimmt.

In [23]:
severity_counts = (
    severity_df
    .groupby("IDpol")
    .size()
    .rename("severity_claim_count")
    .reset_index()
)

severity_counts.head()

,IDpol,severity_claim_count
0,139.0,1
1,190.0,1
2,414.0,1
3,424.0,2
4,463.0,1


### Vergleich der Schadenanzahl

Die Schadenanzahl aus beiden Datensätzen wird pro Police verglichen.

In [24]:
claim_comparison = (
    frequency_df[["IDpol", "ClaimNb"]]
    .merge(
        severity_counts,
        on="IDpol",
        how="left",
    )
)

claim_comparison["severity_claim_count"] = (
    claim_comparison["severity_claim_count"]
    .fillna(0)
    .astype(int)
)

claim_comparison.head(10)

,IDpol,ClaimNb,severity_claim_count
0,1.0,1,0
1,3.0,1,0
2,5.0,1,0
3,10.0,1,0
4,11.0,1,0
5,13.0,1,0
6,15.0,1,0
7,17.0,1,0
8,18.0,1,0
9,21.0,1,0


### Abweichungen zwischen Frequency und Severity

Nun werden Policen mit abweichenden Schadenanzahlen identifiziert.

In [25]:
claim_mismatches = claim_comparison[
    claim_comparison["ClaimNb"]
    != claim_comparison["severity_claim_count"]
]

print("Mismatched policies:", len(claim_mismatches))

claim_mismatches.head()

Mismatched policies: 9117


,IDpol,ClaimNb,severity_claim_count
0,1.0,1,0
1,3.0,1,0
2,5.0,1,0
3,10.0,1,0
4,11.0,1,0


### Art der Abweichungen

Die Abweichungen werden nach ihrer Ursache getrennt ausgewertet.

In [26]:
missing_severity = claim_comparison[
    (claim_comparison["ClaimNb"] > 0)
    & (claim_comparison["severity_claim_count"] == 0)
]

different_counts = claim_comparison[
    (claim_comparison["ClaimNb"] > 0)
    & (claim_comparison["severity_claim_count"] > 0)
    & (
        claim_comparison["ClaimNb"]
        != claim_comparison["severity_claim_count"]
    )
]

print("Total mismatches:", len(claim_mismatches))
print("Claims without severity:", len(missing_severity))
print("Different positive claim counts:", len(different_counts))

Total mismatches: 9117
Claims without severity: 9116
Different positive claim counts: 1


**Ergebnis:** Bei 9.117 Policen stimmen Frequency- und Severity-Daten nicht überein. In 9.116 Fällen fehlt die Severity vollständig; bei einer Police unterscheiden sich die positiven Schadenanzahlen.

In [27]:
different_counts

,IDpol,ClaimNb,severity_claim_count
567077,4158255.0,2,1


**Beispiel:** Für die Police 4158255 werden im Frequency-Datensatz zwei Schäden gemeldet, im Severity-Datensatz ist jedoch nur ein Schaden vorhanden.

## 9. Verwaiste Severity-Einträge

Es wird geprüft, ob Severity-Einträge ohne passende Policy im Frequency-Datensatz vorhanden sind.

In [28]:
orphan_severity = severity_df[
    ~severity_df["IDpol"].isin(frequency_df["IDpol"])
]

print("Orphan severity rows:", len(orphan_severity))
print("Unique orphan policies:", orphan_severity["IDpol"].nunique())

orphan_severity.head()

Orphan severity rows: 195
Unique orphan policies: 6


,IDpol,ClaimAmount
24857,2262511.0,2341.37
24858,2262511.0,1500.00
24863,2262511.0,135.97
24864,2262511.0,1500.00
24865,2262511.0,19461.00


**Ergebnis:** 195 Severity-Einträge gehören zu 6 Policen, die im Frequency-Datensatz nicht vorhanden sind.

## 10. Join-Kardinalität

Die Beziehung zwischen Frequency- und Severity-Datensatz wird überprüft.

In [29]:
joined_df = frequency_df[["IDpol"]].merge(
    severity_df[["IDpol", "ClaimAmount"]],
    on="IDpol",
    how="left",
)

print("Joined rows:", len(joined_df))
print("Frequency rows:", len(frequency_df))
print("Severity rows:", len(severity_df))

Joined rows: 679513
Frequency rows: 678013
Severity rows: 26639


### Erwartete Join-Größe

Die tatsächliche Anzahl der Join-Zeilen wird mit der erwarteten Anzahl verglichen.

In [30]:
frequency_with_severity = frequency_df["IDpol"].isin(
    severity_df["IDpol"]
)

matched_severity = severity_df["IDpol"].isin(
    frequency_df["IDpol"]
)

frequency_without_severity_rows = (~frequency_with_severity).sum()
matched_severity_rows = matched_severity.sum()

expected_joined_rows = (
    frequency_without_severity_rows
    + matched_severity_rows
)

print("Expected joined rows:", expected_joined_rows)
print("Actual joined rows:", len(joined_df))
print(
    "Difference:",
    len(joined_df) - expected_joined_rows
)

Expected joined rows: 679513
Actual joined rows: 679513
Difference: 0


**Ergebnis:** Die tatsächliche Join-Größe entspricht exakt der erwarteten Anzahl. Es wurden keine unerwarteten zusätzlichen Zeilen erzeugt.

## 11. Vergleich mit den automatisierten DQ-Prüfungen

Die manuellen Ergebnisse werden mit den automatisierten Prüfungen aus `checks.py` verglichen.

In [31]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_quality.checks import run_all_checks

In [32]:
automated_results = run_all_checks(
    frequency_df,
    severity_df,
)

automated_results_df = pd.DataFrame(automated_results)

automated_results_df[
    [
        "check_name",
        "dataset_name",
        "status",
        "failed_rows",
        "warning_rows",
    ]
]

,check_name,dataset_name,status,failed_rows,warning_rows
0,missing_values,frequency,PASS,0,0
1,missing_values,severity,PASS,0,0
2,exposure_bounds,frequency,FAIL,1224,0
3,claim_number_bounds,frequency,WARN,0,9
4,claim_amount_bounds,severity,PASS,0,0
5,frequency_severity_consistency,frequency_severity,FAIL,9117,0
6,orphan_severity_records,severity,FAIL,195,0
7,duplicate_records,frequency_severity,WARN,0,255
8,join_cardinality,frequency_severity,PASS,0,0


## 12. Fazit

Die manuellen Prüfungen bestätigen die Ergebnisse der automatisierten Data-Quality-Checks. Die wichtigsten Auffälligkeiten betreffen Exposure-Werte über 1, fehlende Severity-Daten, verwaiste Severity-Einträge und potenzielle Duplikate.